In [1]:
import os
import glob
import re
from nbconvert import HTMLExporter
from traitlets.config import Config
import nbformat

def update_ipynb_links_to_html(html_body):
    """
    Post-process HTML body to replace .ipynb links with .html equivalents, preserving fragments (#anchors).
    
    Args:
        html_body (str): The HTML content as a string.
    
    Returns:
        str: Updated HTML body.
    """
    # Regex to match href=".../file.ipynb#fragment" or href="file.ipynb" (double quotes only)
    # Captures: group(1) = path before .ipynb, group(2) = after .ipynb (e.g., #subheading or query)
    pattern = r'(href=")([^"]*\.)?(?P<before>[^"#\.]*)ipynb(?P<after>#?[^"]*)"'
    def replacer(match):
        # Replace .ipynb with .html, preserving path, before, and after (including #anchor)
        return f'{match.group(1)}{match.group(2)}{match.group("before")}html{match.group("after")}"'
    
    updated_body = re.sub(pattern, replacer, html_body)
    return updated_body

def convert_ipynb_to_html_with_link_updates(directory_path):
    """
    Recursively convert all .ipynb files to HTML, preserving relative images and updating .ipynb links (with anchors) to .html.
    Special handling: Rename "OPEN ME FIRST - README - Main Menu.ipynb" output to root index.html.
    
    Args:
        directory_path (str): Path to the root directory to search for .ipynb files.
    """
    # Find all .ipynb files recursively (sorted for consistent order)
    ipynb_files = sorted(glob.glob(os.path.join(directory_path, '**', '*.ipynb'), recursive=True))
    
    if not ipynb_files:
        print("No .ipynb files found in the directory.")
        return
    
    # Configure exporter for relative paths (no embedding)
    c = Config()
    c.HTMLExporter.embed_images = False
    exporter = HTMLExporter(config=c)
    
    root_dir = os.path.abspath(directory_path)
    target_notebook_name = "OPEN ME FIRST - README - Main Menu.ipynb"  # Exact filename with spaces
    
    for ipynb_path in ipynb_files:
        try:
            # Read the notebook
            with open(ipynb_path, 'r', encoding='utf-8') as f:
                notebook = nbformat.read(f, as_version=4)
            
            # Convert to HTML
            body, resources = exporter.from_notebook_node(notebook)
            
            # Post-process: Update .ipynb links to .html, including anchors
            body = update_ipynb_links_to_html(body)
            
            # Determine output path
            notebook_name = os.path.basename(ipynb_path)
            if notebook_name == target_notebook_name:
                # Special case: Rename to root index.html
                html_filename = 'index.html'
                html_path = os.path.join(root_dir, html_filename)
                resource_dir_name = 'index_files'  # For attachments
                print(f"Special conversion: Renaming to root {html_path}")
            else:
                # Standard: .html in same dir
                html_dir = os.path.dirname(ipynb_path)
                html_filename = os.path.splitext(notebook_name)[0] + '.html'
                html_path = os.path.join(html_dir, html_filename)
                resource_dir_name = f"{os.path.splitext(html_filename)[0]}_files"
            
            # Write updated HTML file
            with open(html_path, 'w', encoding='utf-8') as f:
                f.write(body)
            
            # Extract attachments to _files if present
            if resources and 'files' in resources:
                if notebook_name == target_notebook_name:
                    resource_dir = os.path.join(root_dir, resource_dir_name)
                else:
                    resource_dir = os.path.join(os.path.dirname(html_path), resource_dir_name)
                os.makedirs(resource_dir, exist_ok=True)
                for file_path, content in resources.get('files', {}).items():
                    with open(os.path.join(resource_dir, file_path), 'wb') as res_file:
                        res_file.write(content)
                print(f"Extracted attachments to: {resource_dir}")
            
            print(f"Converted and updated links (with anchors): {ipynb_path} -> {html_path}")
            
        except Exception as e:
            print(f"Error converting {ipynb_path}: {str(e)}")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
convert_ipynb_to_html_with_link_updates(directory)

Converted and updated links (with anchors): .\CAD_Files\3 Port Reservoir\Manufacturing Notes.ipynb -> .\CAD_Files\3 Port Reservoir\Manufacturing Notes.html
Converted and updated links (with anchors): .\ChronoSeq_Overview.ipynb -> .\ChronoSeq_Overview.html
Special conversion: Renaming to root C:\Users\ChronoSeq\ChronoSeq\index.html
Converted and updated links (with anchors): .\OPEN ME FIRST - README - Main Menu.ipynb -> C:\Users\ChronoSeq\ChronoSeq\index.html
Converted and updated links (with anchors): .\convertToHTML.ipynb -> .\convertToHTML.html
Converted and updated links (with anchors): .\instructions_for_assembling_sensirion_flow_sensor.ipynb -> .\instructions_for_assembling_sensirion_flow_sensor.html
Converted and updated links (with anchors): .\instructions_for_assembling_valve_controller.ipynb -> .\instructions_for_assembling_valve_controller.html
Converted and updated links (with anchors): .\instructions_for_assembling_valve_controllerV2.ipynb -> .\instructions_for_assembling_v

C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\instructions_for_assembling_valve_controller_ESP32.ipynb -> .\instructions_for_assembling_valve_controller_ESP32.html
Converted and updated links (with anchors): .\instructions_for_assembling_vortex_relay_controller.ipynb -> .\instructions_for_assembling_vortex_relay_controller.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\instructions_for_assembling_xy_robot_and_ice_boxes.ipynb -> .\instructions_for_assembling_xy_robot_and_ice_boxes.html
Converted and updated links (with anchors): .\instructions_for_assembling_xyz_robot.ipynb -> .\instructions_for_assembling_xyz_robot.html
Converted and updated links (with anchors): .\instructions_for_device_assembly_and_setup.ipynb -> .\instructions_for_device_assembly_and_setup.html
Converted and updated links (with anchors): .\instructions_for_setting_coordinates_for_xyz_robot.ipynb -> .\instructions_for_setting_coordinates_for_xyz_robot.html
Converted and updated links (with anchors): .\instructions_for_setting_up_valves_tubing_and_reservoirs.ipynb -> .\instructions_for_setting_up_valves_tubing_and_reservoirs.html
Converted and updated links (with anchors): .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.ipynb -> .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.html
Converted and up

...

Converted and updated links (with anchors): .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.html
Converted and updated links (with anchors): .\protocol_for_tagmentation_with_KAPA_PCR.ipynb -> .\protocol_for_tagmentation_with_KAPA_PCR.html
Converted and updated links (with anchors): .\protocol_library_preparation_for_dropseq_chronoseq_beads.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads.html
Converted and updated links (with anchors): .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.html
Converted and updated links (with anchors): .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.ipynb -> .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\qPCR_validation_files\qPCR Analysis.ipynb -> .\qPCR_validation_files\qPCR Analysis.html
Converted and updated links (with anchors): .\removeLinksFromNBConvertHTML.ipynb -> .\removeLinksFromNBConvertHTML.html


In [2]:
import os

def generate_sitemap_html(directory_path, output_file='sitemap.html', debug=False):
    """
    Generate a professional HTML site map listing all .html files, categorized by filename patterns and directory structure.
    Includes a new "Main Hardware Pages" section with subheadings for assembly and operation.
    - "instructions*" files under "ChronoSeq Hardware Assembly".
    - "protocol*" files under "ChronoSeq Protocols".
    - Others under "Other Notebooks" (hierarchical by dir, including qPCR_validation_files and _files folders).
    Ignores only ivPID directory and specified basenames: ChronoSeq_Overview, convertToHTML, index, removeLinksFromNBConvertHTML, sitemap.
    
    Args:
        directory_path (str): Path to the root directory to scan for .html files.
        output_file (str): Name of the output HTML file (default: sitemap.html in root).
        debug (bool): If True, print found .html paths (including qPCR_validation_files and _files specifics) to console.
    """
    root_dir = os.path.abspath(directory_path)
    
    # Ignored filenames (case-insensitive, without .html extension)
    ignored_basenames = {
        'chronoseq_overview', 'convertohtml', 'index', 
        'removelinksfromnbconverthtml', 'sitemap'
    }
    
    # Manual main hardware links (adjust paths if not in root)
    main_hardware_links = {
        'assembly': 'instructions_for_device_assembly_and_setup.html',
        'operation': 'protocol_and_software_for_running_chronoseq_device.html'  # Corrected for operation page
    }
    
    # Collect all .html files recursively, excluding ignored
    html_files = {}
    hardware_files = []
    protocol_files = []
    all_found = []  # For debug
    qpcr_files = []  # Specific for qPCR_validation_files debug
    
    for root, dirs, files in os.walk(root_dir):
        rel_root = os.path.relpath(root, root_dir).replace(os.sep, '/')  # Normalize to /
        
        # Skip ivPID directory only
        if 'ivPID' in rel_root.split('/'):
            dirs[:] = []  # Prevent further walking
            continue
        
        # No skipping _files directories - scan inside them for .html
        
        for file in files:
            if file.endswith('.html'):
                # Check for ignored basenames (case-insensitive)
                basename_stripped = os.path.splitext(file)[0].lower()
                full_rel_path = os.path.join(rel_root, file).replace(os.sep, '/')
                if basename_stripped in ignored_basenames:
                    if debug:
                        print(f"Ignored (basename match): {full_rel_path}")
                    continue  # Skip ignored files
                
                all_found.append(full_rel_path)
                basename = file.lower()  # For pattern matching
                
                # Specific debug for qPCR_validation_files
                if 'qpcr_validation_files' in rel_root:
                    qpcr_files.append(full_rel_path)
                
                if basename.startswith('instructions'):
                    hardware_files.append(full_rel_path)
                elif basename.startswith('protocol'):
                    protocol_files.append(full_rel_path)
                else:
                    # For others, group by directory for hierarchy (use / for key)
                    file_dir = rel_root if rel_root != '.' else ''
                    if file_dir not in html_files:
                        html_files[file_dir] = []
                    html_files[file_dir].append(full_rel_path)
    
    # Debug output
    if debug:
        print("All found .html files (excluding ignored, with / paths; now including _files folders):")
        for path in sorted(all_found):
            print(f"  {path}")
        print(f"\nHardware files ({len(hardware_files)}): {sorted(hardware_files)}")
        print(f"Protocol files ({len(protocol_files)}): {sorted(protocol_files)}")
        print(f"Other files by dir: {len(html_files)} directories")
        for dir_key, flist in sorted(html_files.items()):
            if '_files' in dir_key:  # Highlight _files for verification
                print(f"  {dir_key} (from _files folder): {len(flist)} files - {sorted(flist)}")
            else:
                print(f"  {dir_key}: {len(flist)} files - {sorted(flist)[:3]}...")  # Truncate long lists
        if qpcr_files:
            print(f"\nqPCR_validation_files specific files ({len(qpcr_files)}): {sorted(qpcr_files)}")
        else:
            print("\nqPCR_validation_files: No .html files found (check if only .ipynb or excluded). Run 'find qPCR_validation_files -name \"*.html\"' to verify.")
    
    # Sort lists (ensure / paths)
    hardware_files = sorted(hardware_files)
    protocol_files = sorted(protocol_files)
    for dir_key in sorted(html_files.keys()):
        html_files[dir_key] = sorted(html_files[dir_key])
    
    total_files = len(hardware_files) + len(protocol_files) + sum(len(files_list) for files_list in html_files.values())
    
    if total_files == 0:
        print("No .html files found in the directory (excluding ignored paths).")
        return
    
    # Build HTML content with professional styling and new Main Hardware Pages section
    html_content = [
        '<!DOCTYPE html>',
        '<html lang="en">',
        '<head>',
        '    <meta charset="UTF-8">',
        '    <meta name="viewport" content="width=device-width, initial-scale=1.0">',
        '    <title>ChronoSeq Links</title>',
        '    <style>',
        '        :root { --primary-color: #2c5aa0; --secondary-color: #4a90e2; --accent-color: #7ed321; --text-color: #333; --bg-color: #f8f9fa; --card-bg: #ffffff; --shadow: 0 4px 6px rgba(0, 0, 0, 0.1); --border-radius: 8px; --transition: all 0.3s ease; }',
        '        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; line-height: 1.6; color: var(--text-color); background: linear-gradient(135deg, var(--bg-color) 0%, #e3f2fd 100%); margin: 0; padding: 20px; }',
        '        .container { max-width: 1200px; margin: 0 auto; }',
        '        h1 { text-align: center; color: var(--primary-color); font-size: 2.5rem; margin-bottom: 10px; text-shadow: 0 2px 4px rgba(0, 0, 0, 0.1); }',
        '        .subtitle { text-align: center; font-size: 1.1rem; color: #666; margin-bottom: 30px; }',
        '        .total { text-align: center; background: var(--card-bg); padding: 10px; border-radius: var(--border-radius); box-shadow: var(--shadow); display: inline-block; margin-bottom: 30px; font-weight: bold; color: var(--secondary-color); }',
        '        .section { background: var(--card-bg); margin: 20px 0; padding: 25px; border-radius: var(--border-radius); box-shadow: var(--shadow); transition: var(--transition); }',
        '        .section:hover { transform: translateY(-2px); box-shadow: 0 8px 15px rgba(0, 0, 0, 0.15); }',
        '        h2 { color: var(--primary-color); border-bottom: 2px solid var(--secondary-color); padding-bottom: 10px; margin-bottom: 20px; font-size: 1.8rem; }',
        '        h3 { color: var(--secondary-color); margin: 20px 0 15px 0; font-size: 1.3rem; border-left: 4px solid var(--accent-color); padding-left: 10px; }',
        '        ul { list-style: none; padding-left: 0; }',
        '        li { margin: 12px 0; position: relative; padding-left: 25px; }',
        '        li::before { content: "▶"; color: var(--accent-color); font-weight: bold; position: absolute; left: 0; }',
        '        a { text-decoration: none; color: var(--primary-color); font-weight: 500; padding: 8px 12px; background: rgba(74, 144, 226, 0.1); border-radius: 4px; display: inline-block; transition: var(--transition); }',
        '        a:hover { background: var(--secondary-color); color: white; transform: scale(1.05); }',
        '        a:focus { outline: 2px solid var(--accent-color); outline-offset: 2px; }',
        '        .intro { text-align: center; background: rgba(126, 211, 33, 0.1); padding: 20px; border-radius: var(--border-radius); margin-bottom: 30px; font-style: italic; color: #555; }',
        '        @media (max-width: 768px) { body { padding: 10px; } h1 { font-size: 2rem; } .section { padding: 15px; } }',
        '    </style>',
        '</head>',
        '<body>',
        '    <div class="container">',
        '        <h1>ChronoSeq Links</h1>',
        '        <p class="subtitle">Explore the key resources and documentation for the ChronoSeq project</p>',
        '        <p class="total">Total: ' + str(total_files) + ' pages</p>',
        '        <div class="intro">Navigate through hardware assembly, protocols, and analysis notebooks for single-cell RNA sequencing innovation.</div>'
    ]
    
    # New Section: Main Hardware Pages (corrected operation link)
    html_content.extend([
        '        <div class="section">',
        '            <h2>Main Hardware Pages</h2>',
        '            <h3>Hardware Assembly</h3>',
        f'            <ul><li><a href="{main_hardware_links["assembly"]}">View Main Hardware Assembly Page</a></li></ul>',
        '            <h3>Hardware Operation</h3>',
        f'            <ul><li><a href="{main_hardware_links["operation"]}">View Main Hardware Operation Page</a></li></ul>',
        '        </div>'
    ])
    
    # Section: ChronoSeq Hardware Assembly (flat)
    if hardware_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>ChronoSeq Hardware Assembly</h2>',
            '            <ul>'
        ])
        for rel_path in hardware_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for display
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: ChronoSeq Protocols (flat)
    if protocol_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>ChronoSeq Protocols</h2>',
            '            <ul>'
        ])
        for rel_path in protocol_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: Other Notebooks (hierarchical by dir, with /)
    if html_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>Other Notebooks</h2>'
        ])
        sorted_dirs = sorted(html_files.keys())
        current_dir = ""
        for dir_key in sorted_dirs:
            if dir_key != current_dir:
                if current_dir:
                    html_content.append('            </ul>')
                if dir_key:
                    dir_display = dir_key.replace('/', ' > ')
                    html_content.extend([
                        f'            <h3>{dir_display}</h3>',
                        '            <ul>'
                    ])
                else:
                    html_content.extend([
                        '            <h3>Root Directory</h3>',
                        '            <ul>'
                    ])
                current_dir = dir_key
            
            for rel_path in html_files[dir_key]:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for readability
                html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        
        html_content.append('            </ul>')  # Close final ul
        html_content.append('        </div>')
    
    html_content.extend([
        '    </div>',
        '</body>',
        '</html>'
    ])
    
    # Write to sitemap.html in root
    output_path = os.path.join(root_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_content))
    
    print(f"Generated professional sitemap.html with {total_files} HTML files at: {output_path}")
    print("Corrected 'Hardware Operation' link to protocol_and_software_for_running_chronoseq_device.html.")
    print("Styling remains modern and responsive. _files scanned; qPCR_validation_files included under 'Other Notebooks'.")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
generate_sitemap_html(directory, debug=False)  # Set debug=True if needed for verification


Generated professional sitemap.html with 41 HTML files at: C:\Users\ChronoSeq\ChronoSeq\sitemap.html
Corrected 'Hardware Operation' link to protocol_and_software_for_running_chronoseq_device.html.
Styling remains modern and responsive. _files scanned; qPCR_validation_files included under 'Other Notebooks'.


In [3]:
import os

def generate_sitemap_html(directory_path, output_file='sitemap.html', debug=False):
    """
    Generate a professional HTML site map listing all .html files, categorized by filename patterns and directory structure.
    Includes a new "Main Hardware Pages" section with subheadings for assembly and operation.
    - "instructions*" files under "ChronoSeq Hardware Assembly".
    - "protocol*" files under "ChronoSeq Protocols".
    - Others under "Other Notebooks" (hierarchical by dir, including qPCR_validation_files and _files folders).
    Ignores only ivPID directory and specified basenames: ChronoSeq_Overview, convertToHTML, index, removeLinksFromNBConvertHTML, sitemap.
    
    Args:
        directory_path (str): Path to the root directory to scan for .html files.
        output_file (str): Name of the output HTML file (default: sitemap.html in root).
        debug (bool): If True, print found .html paths (including qPCR_validation_files and _files specifics) to console.
    """
    root_dir = os.path.abspath(directory_path)
    
    # Ignored filenames (case-insensitive, without .html extension) - convertToHTML is explicitly here
    ignored_basenames = {
        'chronoseq_overview', 'convertohtml', 'index', 
        'removelinksfromnbconverthtml', 'sitemap'
    }
    
    # Manual main hardware links (adjust paths if not in root)
    main_hardware_links = {
        'assembly': 'instructions_for_device_assembly_and_setup.html',
        'operation': 'protocol_and_software_for_running_chronoseq_device.html'  # Corrected for operation page
    }
    
    # Collect all .html files recursively, excluding ignored
    html_files = {}
    hardware_files = []
    protocol_files = []
    all_found = []  # For debug
    qpcr_files = []  # Specific for qPCR_validation_files debug
    
    for root, dirs, files in os.walk(root_dir):
        rel_root = os.path.relpath(root, root_dir).replace(os.sep, '/')  # Normalize to /
        
        # Skip ivPID directory only
        if 'ivPID' in rel_root.split('/'):
            dirs[:] = []  # Prevent further walking
            continue
        
        # No skipping _files directories - scan inside them for .html
        
        for file in files:
            if file.endswith('.html'):
                # Check for ignored basenames (case-insensitive)
                basename_stripped = os.path.splitext(file)[0].lower()
                full_rel_path = os.path.join(rel_root, file).replace(os.sep, '/')
                if basename_stripped in ignored_basenames:
                    if debug:
                        print(f"Ignored (basename match): {full_rel_path}")
                    continue  # Skip ignored files - this catches convertToHTML.html
                
                all_found.append(full_rel_path)
                basename = file.lower()  # For pattern matching
                
                # Specific debug for qPCR_validation_files
                if 'qpcr_validation_files' in rel_root:
                    qpcr_files.append(full_rel_path)
                
                if basename.startswith('instructions'):
                    hardware_files.append(full_rel_path)
                elif basename.startswith('protocol'):
                    protocol_files.append(full_rel_path)
                else:
                    # For others, group by directory for hierarchy (use / for key)
                    file_dir = rel_root if rel_root != '.' else ''
                    if file_dir not in html_files:
                        html_files[file_dir] = []
                    html_files[file_dir].append(full_rel_path)
    
    # Debug output
    if debug:
        print("All found .html files (excluding ignored like convertToHTML.html, with / paths; now including _files folders):")
        for path in sorted(all_found):
            print(f"  {path}")
        print(f"\nHardware files ({len(hardware_files)}): {sorted(hardware_files)}")
        print(f"Protocol files ({len(protocol_files)}): {sorted(protocol_files)}")
        print(f"Other files by dir: {len(html_files)} directories")
        for dir_key, flist in sorted(html_files.items()):
            if '_files' in dir_key:  # Highlight _files for verification
                print(f"  {dir_key} (from _files folder): {len(flist)} files - {sorted(flist)}")
            else:
                print(f"  {dir_key}: {len(flist)} files - {sorted(flist)[:3]}...")  # Truncate long lists
        if qpcr_files:
            print(f"\nqPCR_validation_files specific files ({len(qpcr_files)}): {sorted(qpcr_files)}")
        else:
            print("\nqPCR_validation_files: No .html files found (check if only .ipynb or excluded). Run 'find qPCR_validation_files -name \"*.html\"' to verify.")
    
    # Sort lists (ensure / paths)
    hardware_files = sorted(hardware_files)
    protocol_files = sorted(protocol_files)
    for dir_key in sorted(html_files.keys()):
        html_files[dir_key] = sorted(html_files[dir_key])
    
    total_files = len(hardware_files) + len(protocol_files) + sum(len(files_list) for files_list in html_files.values())
    
    if total_files == 0:
        print("No .html files found in the directory (excluding ignored paths).")
        return
    
    # Build HTML content with professional styling and new Main Hardware Pages section
    html_content = [
        '<!DOCTYPE html>',
        '<html lang="en">',
        '<head>',
        '    <meta charset="UTF-8">',
        '    <meta name="viewport" content="width=device-width, initial-scale=1.0">',
        '    <title>ChronoSeq Links</title>',
        '    <style>',
        '        :root { --primary-color: #2c5aa0; --secondary-color: #4a90e2; --accent-color: #7ed321; --text-color: #333; --bg-color: #f8f9fa; --card-bg: #ffffff; --shadow: 0 4px 6px rgba(0, 0, 0, 0.1); --border-radius: 8px; --transition: all 0.3s ease; }',
        '        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; line-height: 1.6; color: var(--text-color); background: linear-gradient(135deg, var(--bg-color) 0%, #e3f2fd 100%); margin: 0; padding: 20px; }',
        '        .container { max-width: 1200px; margin: 0 auto; }',
        '        h1 { text-align: center; color: var(--primary-color); font-size: 2.5rem; margin-bottom: 10px; text-shadow: 0 2px 4px rgba(0, 0, 0, 0.1); }',
        '        .subtitle { text-align: center; font-size: 1.1rem; color: #666; margin-bottom: 30px; }',
        '        .total { text-align: center; background: var(--card-bg); padding: 10px; border-radius: var(--border-radius); box-shadow: var(--shadow); display: inline-block; margin-bottom: 30px; font-weight: bold; color: var(--secondary-color); }',
        '        .section { background: var(--card-bg); margin: 20px 0; padding: 25px; border-radius: var(--border-radius); box-shadow: var(--shadow); transition: var(--transition); }',
        '        .section:hover { transform: translateY(-2px); box-shadow: 0 8px 15px rgba(0, 0, 0, 0.15); }',
        '        h2 { color: var(--primary-color); border-bottom: 2px solid var(--secondary-color); padding-bottom: 10px; margin-bottom: 20px; font-size: 1.8rem; }',
        '        h3 { color: var(--secondary-color); margin: 20px 0 15px 0; font-size: 1.3rem; border-left: 4px solid var(--accent-color); padding-left: 10px; }',
        '        ul { list-style: none; padding-left: 0; }',
        '        li { margin: 12px 0; position: relative; padding-left: 25px; }',
        '        li::before { content: "▶"; color: var(--accent-color); font-weight: bold; position: absolute; left: 0; }',
        '        a { text-decoration: none; color: var(--primary-color); font-weight: 500; padding: 8px 12px; background: rgba(74, 144, 226, 0.1); border-radius: 4px; display: inline-block; transition: var(--transition); }',
        '        a:hover { background: var(--secondary-color); color: white; transform: scale(1.05); }',
        '        a:focus { outline: 2px solid var(--accent-color); outline-offset: 2px; }',
        '        .intro { text-align: center; background: rgba(126, 211, 33, 0.1); padding: 20px; border-radius: var(--border-radius); margin-bottom: 30px; font-style: italic; color: #555; }',
        '        @media (max-width: 768px) { body { padding: 10px; } h1 { font-size: 2rem; } .section { padding: 15px; } }',
        '    </style>',
        '</head>',
        '<body>',
        '    <div class="container">',
        '        <h1>ChronoSeq Links</h1>',
        '        <p class="subtitle">Explore the key resources and documentation for the ChronoSeq project</p>',
        '        <p class="total">Total: ' + str(total_files) + ' pages</p>',
        '        <div class="intro">Navigate through hardware assembly, protocols, and analysis notebooks for single-cell RNA sequencing innovation.</div>'
    ]
    
    # New Section: Main Hardware Pages (corrected operation link)
    html_content.extend([
        '        <div class="section">',
        '            <h2>Main Hardware Pages</h2>',
        '            <h3>Hardware Assembly</h3>',
        f'            <ul><li><a href="{main_hardware_links["assembly"]}">View Main Hardware Assembly Page</a></li></ul>',
        '            <h3>Hardware Operation</h3>',
        f'            <ul><li><a href="{main_hardware_links["operation"]}">View Main Hardware Operation Page</a></li></ul>',
        '        </div>'
    ])
    
    # Section: ChronoSeq Hardware Assembly (flat)
    if hardware_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>ChronoSeq Hardware Assembly</h2>',
            '            <ul>'
        ])
        for rel_path in hardware_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for display
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: ChronoSeq Protocols (flat)
    if protocol_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>ChronoSeq Protocols</h2>',
            '            <ul>'
        ])
        for rel_path in protocol_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: Other Notebooks (hierarchical by dir, with /)
    if html_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>Other Notebooks</h2>'
        ])
        sorted_dirs = sorted(html_files.keys())
        current_dir = ""
        for dir_key in sorted_dirs:
            if dir_key != current_dir:
                if current_dir:
                    html_content.append('            </ul>')
                if dir_key:
                    dir_display = dir_key.replace('/', ' > ')
                    html_content.extend([
                        f'            <h3>{dir_display}</h3>',
                        '            <ul>'
                    ])
                else:
                    html_content.extend([
                        '            <h3>Root Directory</h3>',
                        '            <ul>'
                    ])
                current_dir = dir_key
            
            for rel_path in html_files[dir_key]:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for readability
                html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        
        html_content.append('            </ul>')  # Close final ul
        html_content.append('        </div>')
    
    html_content.extend([
        '    </div>',
        '</body>',
        '</html>'
    ])
    
    # Write to sitemap.html in root
    output_path = os.path.join(root_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_content))
    
    print(f"Generated professional sitemap.html with {total_files} HTML files at: {output_path}")
    print("convertToHTML.html is fully excluded via ignored_basenames (no <li> tag generated). Delete old sitemap.html before running.")
    print("Styling remains modern and responsive. _files scanned; qPCR_validation_files included under 'Other Notebooks'.")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
generate_sitemap_html(directory, debug=True)  # Set debug=True to confirm ignored files like convertToHTML.html


Ignored (basename match): ./ChronoSeq_Overview.html
Ignored (basename match): ./index.html
Ignored (basename match): ./removeLinksFromNBConvertHTML.html
Ignored (basename match): ./sitemap.html
All found .html files (excluding ignored like convertToHTML.html, with / paths; now including _files folders):
  ./convertToHTML.html
  ./instructions_for_assembling_sensirion_flow_sensor.html
  ./instructions_for_assembling_valve_controller.html
  ./instructions_for_assembling_valve_controllerV2.html
  ./instructions_for_assembling_valve_controller_ESP32.html
  ./instructions_for_assembling_vortex_relay_controller.html
  ./instructions_for_assembling_xy_robot_and_ice_boxes.html
  ./instructions_for_assembling_xyz_robot.html
  ./instructions_for_device_assembly_and_setup.html
  ./instructions_for_setting_coordinates_for_xyz_robot.html
  ./instructions_for_setting_up_valves_tubing_and_reservoirs.html
  ./protocol_and_software_for_running_chronoseq_device-OriginalDevice.html
  ./protocol_and_softw